# ClearBank - Análise Financeira com Python

Este notebook implementa a solução completa para o desafio de análise de dados da ClearBank.

## Instruções

1. Execute as células em ordem, do início ao fim.
2. A primeira célula gera automaticamente o arquivo `transacoes.csv` com dados de teste.
3. O notebook lê, valida, processa e exporta as métricas financeiras.
4. Ao final, verifique o arquivo `relatorio.json` gerado.

In [1]:
csv_conteudo = """id,data,cliente_id,tipo,valor,descricao,categoria
1,2026-01-05,CLI001,credito,3500.00,Salário janeiro,salario
2,2026-01-12,CLI002,debito,180.50,Supermercado,compra
3,2026-01-20,CLI001,debito,450.00,Aluguel,conta
4,2026-01-25,CLI003,credito,12000.00,Bônus anual,salario
5,2026-02-03,CLI002,debito,200.00,Gasolina,compra
6,2026-02-14,CLI003,credito,15000.00,Transferência recebida,transferencia
7,2026-02-18,CLI002,debito,320.00,Conta de luz,conta
8,2026-02-22,CLI001,debito,89.90,Internet,conta
9,2026-03-01,CLI001,credito,3500.00,Salário março,salario
10,2026-03-10,CLI003,debito,99.90,Streaming,assinatura
11,2026-03-15,CLI004,debito,250.00,Restaurante,compra
12,2026-03-20,CLI002,credito,2800.00,Freelance,salario
13,2026-04-05,CLI001,debito,1200.00,Cartão de crédito,conta
14,2026-04-12,CLI004,credito,5000.00,Venda de equipamento,transferencia
15,2026-04-18,CLI003,debito,75.00,Café,compra
16,2026-04-25,CLI002,debito,600.00,Manutenção carro,compra
17,2026-01-08,CLI005,credito,4200.00,Salário janeiro,salario
18,2026-02-28,CLI005,debito,150.00,Farmácia,compra
19,2026-03-30,CLI004,credito,11000.00,Investimento retorno,transferencia
,2026-02-10,CLI001,debito,300.00,Id vazio,compra
21,2026-02-10,,debito,200.00,Cliente vazio,transferencia
22,2026/03/05,CLI002,credito,500.00,Data inválida,compra
23,2026-04-01,CLI001,deposito,1000.00,Tipo inválido,salario
24,2026-04-08,CLI003,debito,abc,Valor não numérico,compra
25,2026-04-15,CLI002,debito,-50.00,Valor negativo,compra"""

with open('transacoes.csv', 'w', encoding='utf-8') as arquivo:
    arquivo.write(csv_conteudo)

print("Arquivo transacoes.csv gerado com sucesso!")

Arquivo transacoes.csv gerado com sucesso!


In [9]:
import csv
import json
from datetime import datetime, date

In [10]:
LIMITE_SUSPEITO = 10000.00
ARQUIVO_CSV = 'transacoes.csv'
ARQUIVO_JSON = 'relatorio.json'

In [4]:
def ler_transacoes(caminho_csv):
    transacoes_brutas = []
    try:
        with open(caminho_csv, mode='r', encoding='utf-8') as arquivo:
            leitor = csv.DictReader(arquivo)
            for linha in leitor:
                transacoes_brutas.append(linha)
    except FileNotFoundError:
        print(f"Erro: Arquivo '{caminho_csv}' não encontrado.")
    return transacoes_brutas

In [2]:
def validar_data(data_str):
    try:
        return datetime.strptime(data_str.strip(), '%Y-%m-%d').date()
    except ValueError:
        return None


def validar_valor(valor_str):
    try:
        valor = float(valor_str.strip())
        if valor <= 0:
            return None
        return valor
    except ValueError:
        return None


def validar_transacao(registro_csv):
    id_str = registro_csv.get('id', '').strip()
    if not id_str:
        return None
    try:
        id_valor = int(id_str)
    except ValueError:
        return None

    cliente_id = registro_csv.get('cliente_id', '').strip()
    if not cliente_id:
        return None

    data_obj = validar_data(registro_csv.get('data', ''))
    if data_obj is None:
        return None

    tipo = registro_csv.get('tipo', '').strip().lower()
    if tipo not in ('credito', 'debito'):
        return None

    valor = validar_valor(registro_csv.get('valor', ''))
    if valor is None:
        return None

    return {
        'id': id_valor,
        'data': data_obj,
        'cliente_id': cliente_id,
        'tipo': tipo,
        'valor': valor,
        'descricao': registro_csv.get('descricao', '').strip(),
        'categoria': registro_csv.get('categoria', '').strip()
    }

In [5]:
def gerar_relatorio(transacoes_validas):
    resumo_mensal = {}
    transacoes_suspeitas = []
    datas_transacoes = []

    for transacao in transacoes_validas:
        mes_chave = transacao['data'].strftime('%Y-%m')
        valor = transacao['valor']

        if mes_chave not in resumo_mensal:
            resumo_mensal[mes_chave] = {
                'quantidade': 0,
                'total_credito': 0.0,
                'total_debito': 0.0,
                'valores': []
            }

        resumo_mensal[mes_chave]['quantidade'] += 1
        resumo_mensal[mes_chave]['valores'].append(valor)

        if transacao['tipo'] == 'credito':
            resumo_mensal[mes_chave]['total_credito'] += valor
        else:
            resumo_mensal[mes_chave]['total_debito'] += valor

        datas_transacoes.append(transacao['data'])

        if valor > LIMITE_SUSPEITO:
            transacoes_suspeitas.append({
                'id': transacao['id'],
                'cliente_id': transacao['cliente_id'],
                'data': transacao['data'].strftime('%Y-%m-%d'),
                'valor': valor
            })

    for mes in resumo_mensal:
        metricas_mes = resumo_mensal[mes]
        metricas_mes['saldo'] = metricas_mes['total_credito'] - metricas_mes['total_debito']
        metricas_mes['media'] = round(sum(metricas_mes['valores']) / len(metricas_mes['valores']), 2)
        metricas_mes['maior_valor'] = max(metricas_mes['valores'])
        metricas_mes['menor_valor'] = min(metricas_mes['valores'])
        del metricas_mes['valores']

    data_inicial = min(datas_transacoes)
    data_final = max(datas_transacoes)
    dias_entre_extremos = (data_final - data_inicial).days

    return {
        'resumo_mensal': resumo_mensal,
        'transacoes_suspeitas': transacoes_suspeitas,
        'periodo_analisado': {
            'data_inicial': data_inicial.strftime('%Y-%m-%d'),
            'data_final': data_final.strftime('%Y-%m-%d'),
            'dias_entre_extremos': dias_entre_extremos
        }
    }

In [6]:
def salvar_json(dados_relatorio, total_validas, total_invalidas, caminho_json):
    dados_exportacao = {
        'gerado_em': date.today().isoformat(),
        'total_transacoes_validas': total_validas,
        'total_transacoes_invalidas': total_invalidas,
        'resumo_mensal': dados_relatorio['resumo_mensal'],
        'transacoes_suspeitas': dados_relatorio['transacoes_suspeitas'],
        'periodo_analisado': dados_relatorio['periodo_analisado']
    }

    with open(caminho_json, 'w', encoding='utf-8') as arquivo:
        json.dump(dados_exportacao, arquivo, ensure_ascii=False, indent=2)

    print(f"Relatório salvo em: {caminho_json}")

In [7]:
def formatar_moeda(valor):
    return f"R$ {valor:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')


def exibir_relatorio(dados_relatorio, total_lidas, total_validas, total_invalidas):
    print("\n===== RESUMO DA LIMPEZA =====")
    print(f"Total de linhas lidas: {total_lidas}")
    print(f"Linhas válidas: {total_validas}")
    print(f"Linhas inválidas: {total_invalidas}")

    print("\n===== RELATÓRIO MENSAL =====")
    for mes in sorted(dados_relatorio['resumo_mensal'].keys()):
        metricas_mes = dados_relatorio['resumo_mensal'][mes]
        print(f"\nMês: {mes}")
        print(f"  Transações: {metricas_mes['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(metricas_mes['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(metricas_mes['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(metricas_mes['saldo'])}")
        print(f"  Média:         {formatar_moeda(metricas_mes['media'])}")
        print(f"  Maior valor:   {formatar_moeda(metricas_mes['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(metricas_mes['menor_valor'])}")

    print("\n===== TRANSAÇÕES SUSPEITAS =====")
    if dados_relatorio['transacoes_suspeitas']:
        for transacao_suspeita in dados_relatorio['transacoes_suspeitas']:
            print(f"ID: {transacao_suspeita['id']} | Cliente: {transacao_suspeita['cliente_id']} | Data: {transacao_suspeita['data']} | Valor: {formatar_moeda(transacao_suspeita['valor'])}")
    else:
        print("Nenhuma transação suspeita encontrada.")

    periodo = dados_relatorio['periodo_analisado']
    print("\n===== PERÍODO ANALISADO =====")
    print(f"Data inicial: {periodo['data_inicial']}")
    print(f"Data final:   {periodo['data_final']}")
    print(f"Dias entre extremos: {periodo['dias_entre_extremos']}")

In [11]:
transacoes_brutas = ler_transacoes(ARQUIVO_CSV)
total_lidas = len(transacoes_brutas)

transacoes_validas = []
total_invalidas = 0

for linha in transacoes_brutas:
    transacao_limpa = validar_transacao(linha)
    if transacao_limpa is not None:
        transacoes_validas.append(transacao_limpa)
    else:
        total_invalidas += 1

total_validas = len(transacoes_validas)

dados_relatorio = gerar_relatorio(transacoes_validas)

exibir_relatorio(dados_relatorio, total_lidas, total_validas, total_invalidas)

salvar_json(dados_relatorio, total_validas, total_invalidas, ARQUIVO_JSON)


===== RESUMO DA LIMPEZA =====
Total de linhas lidas: 25
Linhas válidas: 19
Linhas inválidas: 6

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações: 5
  Total crédito: R$ 19.700,00
  Total débito:  R$ 630,50
  Saldo:         R$ 19.069,50
  Média:         R$ 4.066,10
  Maior valor:   R$ 12.000,00
  Menor valor:   R$ 180,50

Mês: 2026-02
  Transações: 5
  Total crédito: R$ 15.000,00
  Total débito:  R$ 759,90
  Saldo:         R$ 14.240,10
  Média:         R$ 3.151,98
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 89,90

Mês: 2026-03
  Transações: 5
  Total crédito: R$ 17.300,00
  Total débito:  R$ 349,90
  Saldo:         R$ 16.950,10
  Média:         R$ 3.529,98
  Maior valor:   R$ 11.000,00
  Menor valor:   R$ 99,90

Mês: 2026-04
  Transações: 4
  Total crédito: R$ 5.000,00
  Total débito:  R$ 1.875,00
  Saldo:         R$ 3.125,00
  Média:         R$ 1.718,75
  Maior valor:   R$ 5.000,00
  Menor valor:   R$ 75,00

===== TRANSAÇÕES SUSPEITAS =====
ID: 4 | Cliente: CLI003 | Data: 

In [12]:
with open(ARQUIVO_JSON, 'r', encoding='utf-8') as arquivo:
    conteudo_json = json.load(arquivo)

print(json.dumps(conteudo_json, ensure_ascii=False, indent=2))

{
  "gerado_em": "2026-06-17",
  "total_transacoes_validas": 19,
  "total_transacoes_invalidas": 6,
  "resumo_mensal": {
    "2026-01": {
      "quantidade": 5,
      "total_credito": 19700.0,
      "total_debito": 630.5,
      "saldo": 19069.5,
      "media": 4066.1,
      "maior_valor": 12000.0,
      "menor_valor": 180.5
    },
    "2026-02": {
      "quantidade": 5,
      "total_credito": 15000.0,
      "total_debito": 759.9,
      "saldo": 14240.1,
      "media": 3151.98,
      "maior_valor": 15000.0,
      "menor_valor": 89.9
    },
    "2026-03": {
      "quantidade": 5,
      "total_credito": 17300.0,
      "total_debito": 349.9,
      "saldo": 16950.1,
      "media": 3529.98,
      "maior_valor": 11000.0,
      "menor_valor": 99.9
    },
    "2026-04": {
      "quantidade": 4,
      "total_credito": 5000.0,
      "total_debito": 1875.0,
      "saldo": 3125.0,
      "media": 1718.75,
      "maior_valor": 5000.0,
      "menor_valor": 75.0
    }
  },
  "transacoes_suspeitas": [
 